## Import Data

In [15]:
import pandas as pd
import numpy as np
import os

import pandas as pd
import os

local_file = "recruiting_funnel_data.csv"
csv_url = "https://raw.githubusercontent.com/catisworkingoncode/HR/main/recruiting_funnel_data.csv"

### Scan Through Data

###Scan the Data - to understand what I have in this cvs file.

candidate_id\
requisition_open_date\
application_date\
...\
total_days_to_offer\
total_days_to_hire

In [16]:
df.shape

(10500, 35)

###Understand Dimensions - to understand what I have in this cvs file.

We have 10,500 rows & 35 columns

In [17]:
df.columns.tolist()

['candidate_id',
 'requisition_open_date',
 'application_date',
 'application_year',
 'application_quarter',
 'business_unit',
 'emt',
 'level3_manager',
 'job_title',
 'job_level',
 'tbp_range',
 'tbp_midpoint_usd',
 'recruiter',
 'location',
 'region',
 'source',
 'gender',
 'ethnicity',
 'age_group',
 'stage_phone_screen_date',
 'stage_tech_assessment_date',
 'stage_onsite_date',
 'stage_offer_date',
 'start_date',
 'stage_reached',
 'final_disposition',
 'offer_accepted',
 'rejection_reason',
 'offer_amount_usd',
 'days_applied_to_screen',
 'days_screen_to_tech',
 'days_tech_to_onsite',
 'days_onsite_to_offer',
 'total_days_to_offer',
 'total_days_to_hire']

###See all the Columns - try to have classification

**- Candidate Demographic**

candidate_id\
candidate_name\
location\

**- Job info**

job_title\
job_level\
business_unit\

**- Recruiting info**

recruiter\
source\

**- Funnel dates**

application_date\
phone_screen_date\
technical_date\
onsite_date\
offer_date\
hire_date\

**- Outcomes**

disposition\
offer_status\

**- Derived fields**

days_to_phone_screen\
days_to_offer\
days_to_hire\

In [18]:
df.head(3)

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire
0,CAND-000001,2022-03-07,2022-03-07,2022,Q1 2022,Marketing,CMO,Lena Mueller,Content Strategist,IC5,...,Rejected,NaN,Business need changed,NaN,10.0,13.0,13.0,NaN,NaN,NaN
1,CAND-000002,2022-07-07,2022-07-18,2022,Q3 2022,Sales,CRO,Kevin Brooks,Senior AE,M3,...,Rejected,NaN,Role filled internally,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CAND-000003,2022-03-14,2022-04-04,2022,Q2 2022,Finance,CFO,Sandra Kim,Financial Analyst,IC5,...,Hired,True,NaN,190200.0,15.0,13.0,13.0,10.0,52.0,79.0


In [19]:
df["candidate_id"].nunique()

10500

In [20]:
len(df)

10500

In [21]:
df["candidate_id"].nunique()

10500

Rows:              10,500\
Unique candidate:  10,500


Candidate ID is unique across the dataset, suggesting the dataset is at candidate-record grain.

## Check Data I - Duplicate

###Duplicate - find out the same, repeated items


If we check only if entire rows are duplicated (one row = one candidate), we're having grain problem.

candidate_id | recruiter | source

001          | Alice     | LinkedIn

001          | Bob       | Referral

These are not the same, even though the candidate_id is the same, we still need to know how the internal process proceed better (more efficient & productive)

In [22]:
df.duplicated().sum()

np.int64(0)

In [23]:
df["candidate_id"].duplicated().sum()

np.int64(0)

In [24]:
df[df["candidate_id"].duplicated(keep=False)].sort_values("candidate_id")

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


## Check Data II - Type

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10500 entries, 0 to 10499
Data columns (total 35 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   candidate_id                10500 non-null  object 
 1   requisition_open_date       10500 non-null  object 
 2   application_date            10500 non-null  object 
 3   application_year            10500 non-null  int64  
 4   application_quarter         10500 non-null  object 
 5   business_unit               10500 non-null  object 
 6   emt                         10500 non-null  object 
 7   level3_manager              10500 non-null  object 
 8   job_title                   10500 non-null  object 
 9   job_level                   10500 non-null  object 
 10  tbp_range                   10500 non-null  object 
 11  tbp_midpoint_usd            10500 non-null  int64  
 12  recruiter                   10500 non-null  object 
 13  location                    105

###Covert - change the column to the right type of data
date should be datetime object instead of string object

In [26]:
date_columns = [
    "application_date",
    "requisition_open_date",
    "stage_phone_screen_date",
    "stage_tech_assessment_date",
    "stage_onsite_date",
    "stage_offer_date",
    "start_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df[date_columns].dtypes

,0
application_date,datetime64[ns]
requisition_open_date,datetime64[ns]
stage_phone_screen_date,datetime64[ns]
stage_tech_assessment_date,datetime64[ns]
stage_onsite_date,datetime64[ns]
stage_offer_date,datetime64[ns]
start_date,datetime64[ns]


## Check Data III - Missing

In [27]:
missing = df.isna().sum()

missing

,0
candidate_id,0
requisition_open_date,0
application_date,0
application_year,0
application_quarter,0
business_unit,0
emt,0
level3_manager,0
job_title,0
job_level,0


###Expected missingness - identify those unexpected one

Unexpected NULL, for example:


1. recruiter = NULL
2. candidates_id = NULL
3. job_title = NULL
4. emt/business_unit/level3_manager = NULL (indicate the backend system flow didn't design well, these information shouold be mandatory, automatically filled in when we pull out the job title that the candidate applies)

****by law, company should know the gender, ethnicity, age (over 18+)

What we can expect is:



1. no offer -> no offer date, either candidate has an offer but offer date missing or candidate is hired but recruiter is missing the details
2. no days_tech_to_onsite -> candidate hasn't reached to that stage or the candidate got rejected before the tech interview



In [28]:
display(
    (df.isna().mean() * 100
     ).sort_values(ascending=False
                   ).apply(lambda x: f"{x:.2f}%"))

,0
total_days_to_hire,91.86%
start_date,91.86%
stage_offer_date,89.01%
days_onsite_to_offer,89.01%
offer_accepted,89.01%
total_days_to_offer,89.01%
offer_amount_usd,89.01%
stage_onsite_date,77.49%
days_tech_to_onsite,65.93%
days_screen_to_tech,65.93%


In [29]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
})

display(
    missing_report.sort_values("missing_pct", ascending=False)
    .assign(missing_pct=lambda x: x["missing_pct"].apply(lambda y: f"{y:.2f}%"))
)

,missing_count,missing_pct
total_days_to_hire,9645,91.86%
start_date,9645,91.86%
stage_offer_date,9346,89.01%
days_onsite_to_offer,9346,89.01%
offer_accepted,9346,89.01%
total_days_to_offer,9346,89.01%
offer_amount_usd,9346,89.01%
stage_onsite_date,8136,77.49%
days_tech_to_onsite,6923,65.93%
days_screen_to_tech,6923,65.93%


## Check Data IV - Value Count

In [30]:
df["source"].value_counts(dropna=False)

,count
source,
LinkedIn,3201
Referral,1895
Indeed,1593
Company Website,1225
University / Campus,871
Glassdoor,700
Agency / RPO,628
Handshake,387


In [31]:
df["recruiter"].value_counts(dropna=False)

,count
recruiter,
Drew Martinez,1374
Sam Rivera,1365
Jordan Walsh,1314
Riley Johnson,1309
Morgan Lee,1299
Taylor Kim,1291
Alex Patel,1287
Casey Chen,1261


In [32]:
df["business_unit"].value_counts(dropna=False)

,count
business_unit,
Engineering,2984
Sales,2073
Product,1235
Operations,1083
Marketing,1045
Finance,835
HR,624
Legal,621


In [33]:
df["job_level"].value_counts(dropna=False)

,count
job_level,
IC3,1564
IC4,1496
M2,1410
M3,1364
IC2,1362
IC5,945
M1,682
IC1,680
IC6,602


In [34]:
df["rejection_reason"].value_counts(dropna=False)

,count
rejection_reason,
Resume not qualified,1237
Role filled internally,1230
Withdrew early,1228
Duplicate application,1132
Withdrew,882
NaN,855
No response,528
Skills mismatch,509
Not qualified,391


## Check Data V - Normalization & Chronology

###Categorical - to check whether there's normalization

Categorical fields appear consistently formatted, with no obvious duplicate labels or unexpected categories requiring normalization
(no Engineering/Eng/Engineering Dept in the same column, if it does, it's categorical normalization)

In [35]:
df[date_columns].agg(["min", "max"])

,application_date,requisition_open_date,stage_phone_screen_date,stage_tech_assessment_date,stage_onsite_date,stage_offer_date,start_date
min,2022-01-01,2021-12-03,2022-01-04,2022-01-17,2022-01-28,2022-02-03,2022-02-21
max,2024-12-31,2024-12-30,2025-01-20,2025-02-03,2025-02-19,2025-02-27,2025-04-07


###Date - one of the critical KPIs for Recruiters

1. to check if the dates are reasonable
2. the interview process all happened after application
3. all the following process dates are chronologically


In [36]:
df[
    df["stage_phone_screen_date"] < df["application_date"]
]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


In [37]:
df[
    df["stage_tech_assessment_date"] < df["stage_phone_screen_date"]
]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


In [38]:
df[
    df["stage_onsite_date"] < df["stage_tech_assessment_date"]
]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


In [39]:
df[
    df["stage_offer_date"] < df["stage_onsite_date"]
]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


In [40]:
df[
    df["start_date"] < df["stage_offer_date"]
]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


###My finding:
no chronological inconsistencies were detected in the available stage dates\
(application date - phone screen - tech assessment - onsite - offer - hire/start date)

absence of detected errors doesn't mean it's a perfect data, it just tells me it's synthetic (the data was well designed)*斜體文字*

In [41]:
calculated = (
    df["stage_phone_screen_date"]
    - df["application_date"]
).dt.days

In [42]:
df["days_applied_to_screen"].equals(calculated)

True

In [43]:
difference = (
    df["days_applied_to_screen"]
    - calculated
)

difference.value_counts()

,count
0.0,5257


## Verification I - Data

###Verify I - to check whether the field are valid (trusted)

In [44]:
df["days_applied_to_screen"].describe()


,days_applied_to_screen
count,5257.000000
mean,9.011984
std,4.147266
min,1.000000
25%,6.000000
50%,9.000000
75%,12.000000
max,26.000000


In [45]:
df[df["days_applied_to_screen"] < 0]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


In [46]:
df[df["days_applied_to_screen"] > 365]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


###Verify II - any impossible, unrealistic



1. basic statistics: min/max/median/75%/90%/99%
2. negative
3. over 365 (more like a red flag)



In [47]:
df["final_disposition"].value_counts(dropna=False)

,count
final_disposition,
Rejected,8596
Hired,855
Withdrew,750
Offer Declined,299


In [48]:
df["offer_accepted"].value_counts(dropna=False)

,count
offer_accepted,
NaN,9346
True,855
False,299


In [49]:
rule_1 = (
    (df["final_disposition"] == "Hired") &
    (df["start_date"].isna())
)

df[rule_1]
rule_1.sum()
rule_1.mean() * 100

np.float64(0.0)

## Verfication II - Bz Logic

###Verify III - Bz logic I - hired candidate must have a hire date

If result = 0:
No candidates marked as Hired are missing a hire date

If result = x (any positive #):
x number candidates are marked as Hired but have no recorded hire date

In [50]:
rule_2 = (
    df["start_date"].notna() &
    (df["final_disposition"] != "Hired")
)

df[rule_2]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


###Verify III - Bz logic II - hired date should imply Hired

have to work on both direction:
Hired → hire date exists & hire date exists → Hired

In [51]:
rule_3 = (
    df["start_date"].notna() &
    df["stage_offer_date"].isna()
)

df[rule_3]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


###Verify III - Bz logic III - hired candidate should have offer

If someone was hired with no offer date = potentially suspicious, not 100% wrong,\
there are some possible reasons cause the data show this way:

1. offer date wasn't recorded (missed recorded)
2. offer was handled outside the system
3. the workflow allows direct hiring
4. synthetic data generation didn't enforce the relationship

In [52]:
rule_4 = (
    df["stage_offer_date"].notna() &
    df["stage_onsite_date"].isna()
)

df[rule_4]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


###Verify III - Bz logic IV - have offer but never have onsite

whether this candidate got the offer without showing up onsite interview\
***investigate before delete the data

In [53]:
rule_5 = (
    df["stage_onsite_date"].notna() &
    df["stage_tech_assessment_date"].isna()
)

df[rule_5]


,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


In [54]:
rule_5.sum()

np.int64(0)

###Verify III - Bz logic V - having onsite means having tech assessment

catches missing upstream event rather than just missing data

Finding:\
There are 0 candidates in this dataset who have an onsite_date but are missing a technical_date\
means every candidate with an onsite date also has a technical date

In [55]:
rule_6 = (
    df["stage_tech_assessment_date"].notna() &
    df["stage_phone_screen_date"].isna()
)

df[rule_6]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


###Verify III - Bz logic VI - having tech means having phone screen

Finding:
There are 0 candidates in this dataset who have an tech assessment but are missing a phone screen
means every candidate with an tech assessment also has a phone screen

In [56]:
rule_7 = (
    (df["offer_accepted"] == True) &
    df["stage_offer_date"].isna()
)

df[rule_7]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


###Verify III - Bz logic VII - offer accepted have to have an offer first

Finding: There are 0 candidates in this dataset who have accepted offer but are missing receiving offer letter means every candidate accept has an offer

In [57]:
df["offer_accepted"].value_counts(dropna=False)

,count
offer_accepted,
NaN,9346
True,855
False,299


###Verify III - Bz logic VIII - rejected/withdrawn candidates won't be hired later

if candidate is marked rejected/withdrawn but has a hire date (start date) indicate inconsistency - need to investigate

In [58]:
rule_8 = (
    df["final_disposition"].isin(["Rejected", "Withdrawn"]) &
    df["start_date"].notna()
)

df[rule_8]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


###Verify III - Bz logic IX - date has to be forward

rule 9
- condition#1: stage_phone_screen_date NOT missing
- condition#2: application_date NOT missing
- condition#3: stage_phone_screen_date BEFORE application_date

rule 10
- condition#1: stage_tech_assessment_date NOT missing
- condition#2: stage_phone_screen_date NOT missing
- condition#3: stage_tech_assessment_date BEFORE stage_phone_screen_date


if condition#3 of rule 9 & 10 is true = against chronological rule\
among records where both events exist, identify cases where the phone screen/tech assessment occurred before the application/phone screen

In [59]:
rule_9 = (
    (df["stage_phone_screen_date"].notna()) &
    (df["application_date"].notna()) &
    (df["stage_phone_screen_date"] < df["application_date"])
)

df[rule_9]

,candidate_id,requisition_open_date,application_date,application_year,application_quarter,business_unit,emt,level3_manager,job_title,job_level,...,final_disposition,offer_accepted,rejection_reason,offer_amount_usd,days_applied_to_screen,days_screen_to_tech,days_tech_to_onsite,days_onsite_to_offer,total_days_to_offer,total_days_to_hire


In [60]:
rule_10 = (
    df["stage_tech_assessment_date"].notna() &
    df["stage_phone_screen_date"].notna() &
    (df["stage_tech_assessment_date"] < df["stage_phone_screen_date"])
)

###Verify IV - Bz logic summary

small audit framework
Findings:
this data set has very clean business-rule violation rates, although missing upstream stage dates among downstream-stage candidates warrant investigation

In [61]:
checks = []

checks.append({
    "rule": "Hired candidates must have a start (hired) date",
    "violations": rule_1.sum()
})

checks.append({
    "rule": "Start (hired) date implies final (hired) disposition",
    "violations": rule_2.sum()
})

checks.append({
    "rule": "Hired candidates should have an offer date",
    "violations": rule_3.sum()
})

checks.append({
    "rule": "Offer candidates should have reached onsite interview",
    "violations": rule_4.sum()
})

checks.append({
    "rule": "Onsite candidates should have reached tech assessment",
    "violations": rule_5.sum()
})

checks.append({
    "rule": "Tech assessment candidates should have reached phone screen",
    "violations": rule_6.sum()
})

checks.append({
    "rule": "Accepted offers require an offer date",
    "violations": rule_7.sum()
})

checks.append({
    "rule": "Rejected/withdrawn candidates shouldn't have a start (hired) date",
    "violations": rule_8.sum()
})

audit_df = pd.DataFrame(checks)

audit_df["violation_rate"] = (
    audit_df["violations"] / len(df) * 100
)

audit_df

,rule,violations,violation_rate
0,Hired candidates must have a start (hired) date,0,0.0
1,Start (hired) date implies final (hired) dispo...,0,0.0
2,Hired candidates should have an offer date,0,0.0
3,Offer candidates should have reached onsite in...,0,0.0
4,Onsite candidates should have reached tech ass...,0,0.0
5,Tech assessment candidates should have reached...,0,0.0
6,Accepted offers require an offer date,0,0.0
7,Rejected/withdrawn candidates shouldn't have a...,0,0.0


## Weight Rule

### NOT every rules are weight equally

Classify the rules, since there's always a priority among these judgement

* **Critical**
1. Start (hired) date exists but candidate isn't hired
2. Hired candidate has no start (hired) date
3. Offer accepted without offer
4. Impossible date ordering

* **Warning**
1. Onsite without technical date
2. Offer without onsite date
3. Missing recruiter

* **Expected**
1. Rejected candidate has no offer date
2. Candidate who withdrew has no hire date
3. Candidate who never reached onsite has no onsite date

###Sanity Check

whether reconstructed journey make sense relative to the final disposition

In [62]:
df["stage_reached"] = "Application"

df.loc[df["stage_phone_screen_date"].notna(), "stage_reached"] = "Phone Screen"
df.loc[df["stage_tech_assessment_date"].notna(), "stage_reached"] = "Tech"
df.loc[df["stage_onsite_date"].notna(), "stage_reached"] = "Onsite"
df.loc[df["stage_offer_date"].notna(), "stage_reached"] = "Offer"
df.loc[df["start_date"].notna(), "stage_reached"] = "Hired"

pd.crosstab(
    df["stage_reached"],
    df["final_disposition"]
)

final_disposition,Hired,Offer Declined,Rejected,Withdrew
stage_reached,,,,
Application,0,0,4827,416
Hired,855,0,0,0
Offer,0,299,0,0
Onsite,0,0,1103,107
Phone Screen,0,0,1547,133
Tech,0,0,1119,94


## Final Finding

###Data quality finding:
strong process consistency but ongoing data-quality monitoring remains important


All tested business rules produced 0 violations -> highly consistent stage progression & outcome logic\
a reliable foundation for the **current** funnel analysis


the absence of violations not an evidence that real-world recruiting data will remain equally clean\
This dataset is synthetic and generated under predefined workflow rules\

In production ATS, these can create inconsistencies
- recruiting processes evolve
- new stages and integrations are introduced
- manual data-entry errors


Rather than treating data quality as a one-time cleaning task\
a recruiting platform should continuously monitor to downstream analytics remain trustworthy
- event completeness
- stage chronology
- workflow anomalies


## Stage Flag - Candidate Application Stage

###Candidates application stage

use boolean to list out every candidates' stage

In [63]:
df["reached_application"] = df["application_date"].notna()
df["reached_phone_screen"] = df["stage_phone_screen_date"].notna()
df["reached_tech"] = df["stage_tech_assessment_date"].notna()
df["reached_onsite"] = df["stage_onsite_date"].notna()
df["reached_offer"] = df["stage_offer_date"].notna()
df["reached_hired"] = df["start_date"].notna()

In [64]:
df[
    [
        "candidate_id",
        "reached_application",
        "reached_phone_screen",
        "reached_tech",
        "reached_onsite",
        "reached_offer",
        "reached_hired"
    ]
].head()

,candidate_id,reached_application,reached_phone_screen,reached_tech,reached_onsite,reached_offer,reached_hired
0,CAND-000001,True,True,True,True,False,False
1,CAND-000002,True,False,False,False,False,False
2,CAND-000003,True,True,True,True,True,True
3,CAND-000004,True,False,False,False,False,False
4,CAND-000005,True,True,False,False,False,False


###Candidates count in each stage

use .sum() to summarize the head count (funnel counts) of each stage\
the most important thing is
- how many candidates we hired at the end
- what's the hired conversion rate (from application to hired stage)

Finding:
- hired conversion rate:\
855 out of 10,500 applicants were ultimately hired, giving an application-to-hire conversion rate of 8.1%
- overall funnel conversion rate:\
2,364 out of 10,500 applicants reached to onsite interview, applicantion -to-onsite conversion rate of 22.5%

In [65]:
funnel_counts = {
    "Application": df["reached_application"].sum(),
    "Phone Screen": df["reached_phone_screen"].sum(),
    "Tech": df["reached_tech"].sum(),
    "Onsite": df["reached_onsite"].sum(),
    "Offer": df["reached_offer"].sum(),
    "Hired": df["reached_hired"].sum()
}

funnel_counts

{'Application': np.int64(10500),
 'Phone Screen': np.int64(5257),
 'Tech': np.int64(3577),
 'Onsite': np.int64(2364),
 'Offer': np.int64(1154),
 'Hired': np.int64(855)}

In [66]:
df["reached_hired"].sum()

np.int64(855)

In [67]:
funnel_df = pd.DataFrame(
    list(funnel_counts.items()),
    columns=["stage", "candidates"]
)

funnel_df

,stage,candidates
0,Application,10500
1,Phone Screen,5257
2,Tech,3577
3,Onsite,2364
4,Offer,1154
5,Hired,855


Finding:


- drop-off conversion rate (stage to stage):\
Application → Phone Screen: 1 - (5257/10500) = 0.4993\
Phone Screen → Tech: 1 - (3577/5257) = 0.3195\
Tech → Onsite: 1 - (2364/3577) = 0.3391\
Onsite → Offer: 1 - (1154/2364) = 0.5118\
Offer → Hired: 1 - (855/1154) = 0.2590
- The biggest drop-off session: Onsite → Offer\
the result only tell us where candidates exit and that's where we need to investigate the "why"


In [68]:
#overall conversion from application
funnel_df["conversion_from_application"] = (
    funnel_df["candidates"]
    / funnel_df.loc[0, "candidates"]
)

funnel_df["conversion_from_application_pct"] = (
    funnel_df["conversion_from_application"] * 100
).round(1)

#stage-to-stage conversion
funnel_df["stage_conversion"] = (
    funnel_df["candidates"]
    / funnel_df["candidates"].shift(1) #.shift(1) creates "previous stage" reference
)

#percentage
funnel_df["stage_conversion_pct"] = (
    funnel_df["stage_conversion"] * 100
).round(1)

#drop-off rate
funnel_df["candidate_dropoff"] = (
    funnel_df["candidates"].shift(1)
    - funnel_df["candidates"]
)

funnel_df["dropoff_rate"] = (
    1 - funnel_df["stage_conversion"]
)

funnel_df["dropoff_rate_pct"] = (
    funnel_df["dropoff_rate"] * 100
).round(1)

#display as XX.XX%
funnel_df["conversion_from_application_pct"] = (
    funnel_df["conversion_from_application_pct"].map(lambda x: f"{x:.2f}%")
)

funnel_df["stage_conversion_pct"] = (
    funnel_df["stage_conversion_pct"].map(lambda x: f"{x:.2f}%")
)

funnel_df["dropoff_rate_pct"] = (
    funnel_df["dropoff_rate_pct"].map(lambda x: f"{x:.2f}%")
)

display_df = funnel_df[
    [
        "conversion_from_application_pct",
        "stage_conversion_pct",
        "dropoff_rate_pct"
    ]
]

display_df

,conversion_from_application_pct,stage_conversion_pct,dropoff_rate_pct
0,100.00%,nan%,nan%
1,50.10%,50.10%,49.90%
2,34.10%,68.00%,32.00%
3,22.50%,66.10%,33.90%
4,11.00%,48.80%,51.20%
5,8.10%,74.10%,25.90%


###Candidates application stage - Finding Summary



* **Overall funnel**\
out of 10,500 applications, 855 candidates were ultimately hired,\
resulting in an overall application-to-hire conversion rate of 8.1%\
* **Largest transition loss**\
the largest absolute candidate drop-off occurred between application to phone screen\
where the funnel decreased 5239 (10500-5257) candidates\
the largest proportional drop-off occurred between onsite and offer\
where the funnel decreased 51.2% candidates
* **Be aware of**\
can't be interpreted as process friction, need to consider plenty possible reasons (operational delays, candidates withdrawal, source quality...etc)



## Identify & Find Out Why - Candidate Stage

### Check chronology of each candidate stage

In [69]:
df["reached_application"] = df["application_date"].notna()
df["reached_phone_screen"] = df["stage_phone_screen_date"].notna()
df["reached_tech"] = df["stage_tech_assessment_date"].notna()
df["reached_onsite"] = df["stage_onsite_date"].notna()
df["reached_offer"] = df["stage_offer_date"].notna()
df["reached_hired"] = df["start_date"].notna()

df[
    [
        "reached_application",
        "reached_phone_screen",
        "reached_tech",
        "reached_onsite",
        "reached_offer",
        "reached_hired"
    ]
].head(10)

,reached_application,reached_phone_screen,reached_tech,reached_onsite,reached_offer,reached_hired
0,True,True,True,True,False,False
1,True,False,False,False,False,False
2,True,True,True,True,True,True
3,True,False,False,False,False,False
4,True,True,False,False,False,False
5,True,True,False,False,False,False
6,True,False,False,False,False,False
7,True,False,False,False,False,False
8,True,True,True,False,False,False
9,True,True,True,False,False,False


### How many candidates reached at X stage? (How far did candidates progress?)


In [70]:
df[
    [
        "reached_application",
        "reached_phone_screen",
        "reached_tech",
        "reached_onsite",
        "reached_offer",
        "reached_hired"
    ]
].sum()

,0
reached_application,10500
reached_phone_screen,5257
reached_tech,3577
reached_onsite,2364
reached_offer,1154
reached_hired,855


In [71]:
invalid_funnel = (
    (~df["reached_phone_screen"] & df["reached_tech"]) |
    (~df["reached_tech"] & df["reached_onsite"]) |
    (~df["reached_onsite"] & df["reached_offer"]) |
    (~df["reached_offer"] & df["reached_hired"])
)

invalid_funnel.sum()

np.int64(0)

###Where did each candidate's journey stop? (Where did they stop?)

**it doesn't mean that everyone who reached this stage\
it's the furthest/final stage that the candidate reached\
\
**stage**\
Application 5,257\
**stage_reached**\
Application = 5,243 (10,500 - 5,257 = 5,243)\
Interpretation: people who applied but never reached phone screen

In [72]:
df["stage_reached"].value_counts()

,count
stage_reached,
Application,5243
Phone Screen,1680
Tech,1213
Onsite,1210
Hired,855
Offer,299


The independently reconstructed cumulative funnel aligns exactly with the dataset's stage_reached distribution. The number of candidates stopping at each stage equals the difference between candidates reaching that stage and the next stage. This indicates that stage_reached functions as a terminal/furthest-stage classification rather than a cumulative stage count.

###What happened to candidates at each stage?


In [73]:
df["stage_reached"].value_counts()
df["final_disposition"]

,final_disposition
0,Rejected
1,Rejected
2,Hired
3,Rejected
4,Rejected
...,...
10495,Rejected
10496,Rejected
10497,Rejected
10498,Rejected


###Outcome table

In [74]:
outcome_by_stage = pd.crosstab(
    df["stage_reached"],
    df["final_disposition"]
)

outcome_by_stage

final_disposition,Hired,Offer Declined,Rejected,Withdrew
stage_reached,,,,
Application,0,0,4827,416
Hired,855,0,0,0
Offer,0,299,0,0
Onsite,0,0,1103,107
Phone Screen,0,0,1547,133
Tech,0,0,1119,94


###Outcome percentages

In [75]:
outcome_by_stage_pct = pd.crosstab(
    df["stage_reached"],
    df["final_disposition"],
    normalize="index"
) * 100

outcome_by_stage_pct.round(1)

final_disposition,Hired,Offer Declined,Rejected,Withdrew
stage_reached,,,,
Application,0.0,0.0,92.1,7.9
Hired,100.0,0.0,0.0,0.0
Offer,0.0,100.0,0.0,0.0
Onsite,0.0,0.0,91.2,8.8
Phone Screen,0.0,0.0,92.1,7.9
Tech,0.0,0.0,92.3,7.7


In [76]:
normalize="index"

###Withdrawal rate

recruiter reject a candidate can be an expected selection outcome\
candidate withdraw can potentially indicate:

- slow process
- poor candidate experience
- competing offer
- compensation expectations
- lack of interest
- poor communication
- personal reasons

we don't know which one but worth to investigate

In [77]:
withdrawal_rate_by_stage = (
    df.groupby("stage_reached")["final_disposition"]
      .apply(lambda x: (x == "Withdrawn").mean() * 100)
      .round(1)
)

withdrawal_rate_by_stage

,final_disposition
stage_reached,
Application,0.0
Hired,0.0
Offer,0.0
Onsite,0.0
Phone Screen,0.0
Tech,0.0


###Rejected rate

In [78]:
rejection_rate_by_stage = (
    df.groupby("stage_reached")["final_disposition"]
      .apply(lambda x: (x == "Rejected").mean() * 100)
      .round(1)
)

rejection_rate_by_stage

,final_disposition
stage_reached,
Application,92.1
Hired,0.0
Offer,0.0
Onsite,91.2
Phone Screen,92.1
Tech,92.3


## Potential Why I - Funnel bottleneck (stage conversion)

In [79]:
stages = [
    ("Application", "reached_application"),
    ("Phone Screen", "reached_phone_screen"),
    ("Tech", "reached_tech"),
    ("Onsite", "reached_onsite"),
    ("Offer", "reached_offer"),
    ("Hired", "reached_hired")
]

stage_counts = {
    name: df[col].sum()
    for name, col in stages
}

conversion = []

for i in range(1, len(stages)):
    previous_stage = stages[i-1][0]
    current_stage = stages[i][0]

    previous_count = stage_counts[previous_stage]
    current_count = stage_counts[current_stage]

    conversion.append({
        "from_stage": previous_stage,
        "to_stage": current_stage,
        "previous_count": previous_count,
        "current_count": current_count,
        "conversion_rate": current_count / previous_count
    })

conversion_df = pd.DataFrame(conversion)

conversion_df["conversion_rate_pct"] = (
    conversion_df["conversion_rate"] * 100
).round(1)

conversion_df

,from_stage,to_stage,previous_count,current_count,conversion_rate,conversion_rate_pct
0,Application,Phone Screen,10500,5257,0.500667,50.1
1,Phone Screen,Tech,5257,3577,0.680426,68.0
2,Tech,Onsite,3577,2364,0.660889,66.1
3,Onsite,Offer,2364,1154,0.488156,48.8
4,Offer,Hired,1154,855,0.740901,74.1


### Funnel bottleneck

Only around 48.8% of candidates reaching onsite progressed to an offer, making Onsite → Offer the weakest stage-to-stage conversion in the reconstructed funnel

In [80]:
conversion_df.loc[
    conversion_df["conversion_rate"].idxmin()
]

,3
from_stage,Onsite
to_stage,Offer
previous_count,2364
current_count,1154
conversion_rate,0.488156
conversion_rate_pct,48.8


###Finding I:

potential reasons casue the problem:
*  Healthy filtering - if onsite interviews are intentionally rigorou, then 48.8% conversion might be acceptable
*  Interview inconsistency - interview feedback / scorecard quality
*  Poor candidate qualification upstream - recruiters aren't efficiently identifying strong candidates earlier
*  Candidate experience problem - stage velocity / candidate experience problem
*  Hiring manager calibration - intake / scorecard / alignment proble



In [81]:
df["time_application_to_phone"] = (
    df["stage_phone_screen_date"] -
    df["application_date"]
).dt.days

df["time_phone_to_tech"] = (
    df["stage_tech_assessment_date"] -
    df["stage_phone_screen_date"]
).dt.days

df["time_tech_to_onsite"] = (
    df["stage_onsite_date"] -
    df["stage_tech_assessment_date"]
).dt.days

df["time_onsite_to_offer"] = (
    df["stage_offer_date"] -
    df["stage_onsite_date"]
).dt.days

df["time_offer_to_hire"] = (
    df["start_date"] -
    df["stage_offer_date"]
).dt.days

In [82]:
velocity_cols = [
    "time_application_to_phone",
    "time_phone_to_tech",
    "time_tech_to_onsite",
    "time_onsite_to_offer",
    "time_offer_to_hire"
]

df[velocity_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
time_application_to_phone,5257.0,9.011984,4.147266,1.0,6.0,9.0,12.0,26.0
time_phone_to_tech,3577.0,13.289908,5.609305,1.0,9.0,13.0,17.0,35.0
time_tech_to_onsite,2364.0,16.108714,7.041473,1.0,11.0,16.0,21.0,40.0
time_onsite_to_offer,1154.0,6.365685,2.823160,1.0,5.0,7.0,8.0,15.0
time_offer_to_hire,855.0,18.665497,9.408398,1.0,12.0,19.0,25.0,49.0


###Finding II - Recruiting is a ~2-month journey


Recruiting funnel shows meaningful variation in stage velocity among 855 successful hires\
median time from application to start was 63 days (50% of hires occured between 54 and 72 days)\
The process ranged from 23 to 111 days - indicating while the typical hiring journey is relatively consistent around 2 months\
(a meaningful group of candidates experiences substantially longer cycle)


- Tech → Onsite: \
16 days of median cycle time\
25% of candidates taking > 21 days\
distribution extending to 40 days\
  A potential operational bottleneck and warrants further investigation
  - scheduling
  - assessment completion
  - interviewer feedback
  - candidate response time
- Offer → Hire:\
19 days (the longest median)\
std 9.4 days, P75 25 days, max 49 days (the highest variability)\
  this metric is NOT entirely as recruiter/process friction, external impact factors (may contribute substantially)
  - notice periods
  - relocation
  - candidate-specific constraints
- Conculsion:\
NOT just simply optimizing for lower average cycle time\
recruiting system should
  - help recruiters identify where & why individual candidates are delayed
  - distinguish controllable process delays from external constraints
  - surface candidates at risk of stalling



In [83]:
df["time_to_hire"] = (
    df["start_date"] - df["application_date"]
).dt.days

In [84]:
df["time_to_hire"].describe()

,time_to_hire
count,855.000000
mean,63.436257
std,14.051023
min,23.000000
25%,54.000000
50%,63.000000
75%,72.000000
max,111.000000


### FYI: coefficient of variation

The hiring process has noticeable but not extreme variability, the opportunity may lie in identifying the tail of delayed candidates rather than optimizing every candidate toward an artificially low average

In [85]:
df["time_to_hire"].std() / df["time_to_hire"].mean()

np.float64(0.22149829154582973)

## Priority

In [86]:
df.columns.tolist()

['candidate_id',
 'requisition_open_date',
 'application_date',
 'application_year',
 'application_quarter',
 'business_unit',
 'emt',
 'level3_manager',
 'job_title',
 'job_level',
 'tbp_range',
 'tbp_midpoint_usd',
 'recruiter',
 'location',
 'region',
 'source',
 'gender',
 'ethnicity',
 'age_group',
 'stage_phone_screen_date',
 'stage_tech_assessment_date',
 'stage_onsite_date',
 'stage_offer_date',
 'start_date',
 'stage_reached',
 'final_disposition',
 'offer_accepted',
 'rejection_reason',
 'offer_amount_usd',
 'days_applied_to_screen',
 'days_screen_to_tech',
 'days_tech_to_onsite',
 'days_onsite_to_offer',
 'total_days_to_offer',
 'total_days_to_hire',
 'reached_application',
 'reached_phone_screen',
 'reached_tech',
 'reached_onsite',
 'reached_offer',
 'reached_hired',
 'time_application_to_phone',
 'time_phone_to_tech',
 'time_tech_to_onsite',
 'time_onsite_to_offer',
 'time_offer_to_hire',
 'time_to_hire']

### Prioritize


* #1 source
  - source yield rate
  - source → hire conversion
  - candidate quality
  - sourcing channel ROI
  - GTM/product opportunities
* #2 job_level
  - lower conversion
  - longer Time-to-Hire
  - more offer declines
  - longer offer → hire time
* #3 recruiter
  - recruiter workload
  - operational efficiency
  - process consistency
  - bottleneck detection
  - coaching opportunities
  - whether the recruiting system gives recruiters enough visibility



##Priority I - Source

### Priority I - Source (application volume vs. hiring output)

Referral appears to produce higher-quality candidates but need to determine whether this advantage persists after accounting for role mix before recommending a sourcing strategy

In [87]:
df["source"].value_counts()

,count
source,
LinkedIn,3201
Referral,1895
Indeed,1593
Company Website,1225
University / Campus,871
Glassdoor,700
Agency / RPO,628
Handshake,387


In [88]:
df.groupby("source")["reached_hired"].sum().sort_values(ascending=False)

,reached_hired
source,
LinkedIn,225
Referral,209
Indeed,113
Company Website,104
University / Campus,85
Glassdoor,43
Agency / RPO,40
Handshake,36


### Finding I: highlight of the source

3 sources that worth to investigate:
- Referral\
the strongest-performing source (11.03%) - higher candidate yield\
Question - whether it reflects
  - higher candidate quality
  - differences in role mix, candidate seniority or hiring-process treatment
- LinkedIn\
important acquisition channel by volume but low application-to-hire conversion\
potential inefficiency in downstream recruiting effort\
Question - are recruiters spending disproportionate time processing LinkedIn applicants who ultimately don't progress?
- Campus + Handshake\
relatively strong application-to-hire yield (~9.6%)\
campus-oriented channels may be an efficient source of talent despite generating less absolute volume than LinkedIn\
Question - what kind of job level and role type are the major hired?

In [89]:
source_funnel = (
    df.groupby("source")
      .agg(
          applications=("reached_application", "sum"),
          hires=("reached_hired", "sum")
      )
)

source_funnel["hire_conversion_pct"] = (
    source_funnel["hires"] /
    source_funnel["applications"] * 100
).round(2)

source_funnel.sort_values("hire_conversion_pct", ascending=False)

,applications,hires,hire_conversion_pct
source,,,
Referral,1895,209,11.03
University / Campus,871,85,9.76
Handshake,387,36,9.30
Company Website,1225,104,8.49
Indeed,1593,113,7.09
LinkedIn,3201,225,7.03
Agency / RPO,628,40,6.37
Glassdoor,700,43,6.14


### Finding II: efficiency x contribution

Efficiency - How likely is an application to become a hire?\
Contribution - How much of the organization's hiring does this channel actually produce?\



*  high yield + meaningful volume: strategic channel\
Referral
*  high yield + low volume: scaling opportunity\
Handshake
*  high volume + low yield: efficiency problem\
LinkedIn
*  low volume + low yield: deprioritization candidate\
Glassdoor *** don't cutting it now



In [90]:
source_funnel["hire_share_pct"] = (
    source_funnel["hires"] /
    source_funnel["hires"].sum() * 100
).round(2)

source_funnel["application_share_pct"] = (
    source_funnel["applications"] /
    source_funnel["applications"].sum() * 100
).round(2)

source_funnel.sort_values("hires", ascending=False)

,applications,hires,hire_conversion_pct,hire_share_pct,application_share_pct
source,,,,,
LinkedIn,3201,225,7.03,26.32,30.49
Referral,1895,209,11.03,24.44,18.05
Indeed,1593,113,7.09,13.22,15.17
Company Website,1225,104,8.49,12.16,11.67
University / Campus,871,85,9.76,9.94,8.30
Glassdoor,700,43,6.14,5.03,6.67
Agency / RPO,628,40,6.37,4.68,5.98
Handshake,387,36,9.30,4.21,3.69


##Priority II - Job Level

In [91]:
df["job_level"].value_counts()

,count
job_level,
IC3,1564
IC4,1496
M2,1410
M3,1364
IC2,1362
IC5,945
M1,682
IC1,680
IC6,602


In [92]:
# total application from start
source_job_volume = pd.crosstab(
    df["source"],
    df["job_level"]
)

source_job_volume

job_level,IC1,IC2,IC3,IC4,IC5,IC6,M1,M2,M3,M4
source,,,,,,,,,,
Agency / RPO,40,76,90,106,53,37,35,83,77,31
Company Website,79,163,193,171,101,75,79,155,165,44
Glassdoor,36,95,96,106,69,30,45,92,106,25
Handshake,19,48,56,53,40,26,30,45,50,20
Indeed,110,214,233,207,144,88,113,214,197,73
LinkedIn,218,409,493,468,277,179,204,424,414,115
Referral,128,233,277,277,176,110,116,268,248,62
University / Campus,50,124,126,108,85,57,60,129,107,25


In [93]:
# total hired counts
source_job_hires = pd.crosstab(
    df["source"],
    df["job_level"],
    values=df["reached_hired"],
    aggfunc="sum"
)

source_job_hires

job_level,IC1,IC2,IC3,IC4,IC5,IC6,M1,M2,M3,M4
source,,,,,,,,,,
Agency / RPO,3,5,5,9,4,1,2,4,7,0
Company Website,13,14,12,10,11,8,6,9,15,6
Glassdoor,4,8,8,7,3,1,0,5,4,3
Handshake,1,6,5,8,4,2,0,3,6,1
Indeed,9,14,14,10,10,8,10,19,13,6
LinkedIn,17,28,36,33,17,16,16,28,26,8
Referral,11,29,37,28,20,10,12,31,24,7
University / Campus,4,8,12,13,6,7,6,15,11,3


In [94]:
source_job_conversion = (
    source_job_hires /
    source_job_volume
) * 100

source_job_conversion.round(2)

job_level,IC1,IC2,IC3,IC4,IC5,IC6,M1,M2,M3,M4
source,,,,,,,,,,
Agency / RPO,7.50,6.58,5.56,8.49,7.55,2.70,5.71,4.82,9.09,0.00
Company Website,16.46,8.59,6.22,5.85,10.89,10.67,7.59,5.81,9.09,13.64
Glassdoor,11.11,8.42,8.33,6.60,4.35,3.33,0.00,5.43,3.77,12.00
Handshake,5.26,12.50,8.93,15.09,10.00,7.69,0.00,6.67,12.00,5.00
Indeed,8.18,6.54,6.01,4.83,6.94,9.09,8.85,8.88,6.60,8.22
LinkedIn,7.80,6.85,7.30,7.05,6.14,8.94,7.84,6.60,6.28,6.96
Referral,8.59,12.45,13.36,10.11,11.36,9.09,10.34,11.57,9.68,11.29
University / Campus,8.00,6.45,9.52,12.04,7.06,12.28,10.00,11.63,10.28,12.00


### Finding I:

The source x job-level analysis doesn't show a clear relationship between job seniority and sourcing channel effectiveness\
Referral performs well across several levels but the highest conversion rates are distributed across multiple sources and job levels\

High conversion rate ≠ automatically the best sourcing strategy\

In [95]:
source_job_summary = pd.DataFrame({
    "applications": source_job_volume.stack(),
    "hires": source_job_hires.stack(),
    "conversion_pct": source_job_conversion.stack()
}).reset_index()

source_job_summary.columns = [
    "source",
    "job_level",
    "applications",
    "hires",
    "conversion_pct"
]


source_job_summary.sort_values(
    "conversion_pct",
    ascending=False
)
source_job_summary

,source,job_level,applications,hires,conversion_pct
0,Agency / RPO,IC1,40,3,7.500000
1,Agency / RPO,IC2,76,5,6.578947
2,Agency / RPO,IC3,90,5,5.555556
3,Agency / RPO,IC4,106,9,8.490566
4,Agency / RPO,IC5,53,4,7.547170
...,...,...,...,...,...
75,University / Campus,IC6,57,7,12.280702
76,University / Campus,M1,60,6,10.000000
77,University / Campus,M2,129,15,11.627907
78,University / Campus,M3,107,11,10.280374


In [96]:
source_job_summary["volume_flag"] = np.where(
    source_job_summary["applications"] < 50,
    "Low volume",
    "Sufficient volume"
)

source_job_summary

,source,job_level,applications,hires,conversion_pct,volume_flag
0,Agency / RPO,IC1,40,3,7.500000,Low volume
1,Agency / RPO,IC2,76,5,6.578947,Sufficient volume
2,Agency / RPO,IC3,90,5,5.555556,Sufficient volume
3,Agency / RPO,IC4,106,9,8.490566,Sufficient volume
4,Agency / RPO,IC5,53,4,7.547170,Sufficient volume
...,...,...,...,...,...,...
75,University / Campus,IC6,57,7,12.280702,Sufficient volume
76,University / Campus,M1,60,6,10.000000,Sufficient volume
77,University / Campus,M2,129,15,11.627907,Sufficient volume
78,University / Campus,M3,107,11,10.280374,Sufficient volume


### Finding II:

Job level does not appear to be a major driver of hiring conversion

Across 10 job levels, application-to-hire conversion rate from 7.62% to 9.12% no clear relationship between seniority & conversion\
Optimize recruiting performance solely by job seniority is not sufficient\
sourcing channel may have more meaningful differences when evaluated against both conversion and candidate volume

In [97]:
job_level_summary = (
    df.groupby("job_level")
      .agg(
          applications=("candidate_id", "count"),
          hires=("reached_hired", "sum")
      )
)

job_level_summary["conversion_pct"] = (
    job_level_summary["hires"]
    / job_level_summary["applications"]
    * 100
).round(2)

job_level_summary

,applications,hires,conversion_pct
job_level,,,
IC1,680,62,9.12
IC2,1362,112,8.22
IC3,1564,129,8.25
IC4,1496,118,7.89
IC5,945,75,7.94
IC6,602,53,8.80
M1,682,52,7.62
M2,1410,114,8.09
M3,1364,106,7.77


### Categorize for Visualization

define high and low (channels relative to each other)

- above median volume → High Volume
- below median volume → Low Volume
- above median conversion → High Conversion
- below median conversion → Low Conversion


In [99]:
volume_threshold = source_job_summary["applications"].median()
conversion_threshold = source_job_summary["conversion_pct"].median()

source_job_summary["quadrant"] = np.select(
    [
        (source_job_summary["applications"] >= volume_threshold) &
        (source_job_summary["conversion_pct"] >= conversion_threshold),

        (source_job_summary["applications"] >= volume_threshold) &
        (source_job_summary["conversion_pct"] < conversion_threshold),

        (source_job_summary["applications"] < volume_threshold) &
        (source_job_summary["conversion_pct"] >= conversion_threshold),

        (source_job_summary["applications"] < volume_threshold) &
        (source_job_summary["conversion_pct"] < conversion_threshold)
    ],
    [
        "High volume + High conversion",
        "High volume + Low conversion",
        "Low volume + High conversion",
        "Low volume + Low conversion"
    ],
    default=""
)

In [100]:
source_summary = (
    df.groupby("source")
      .agg(
          applications=("candidate_id", "count"),
          hires=("reached_hired", "sum")
      )
      .reset_index()
)

source_summary["conversion_pct"] = (
    source_summary["hires"]
    / source_summary["applications"]
    * 100
).round(2)

source_summary

,source,applications,hires,conversion_pct
0,Agency / RPO,628,40,6.37
1,Company Website,1225,104,8.49
2,Glassdoor,700,43,6.14
3,Handshake,387,36,9.30
4,Indeed,1593,113,7.09
5,LinkedIn,3201,225,7.03
6,Referral,1895,209,11.03
7,University / Campus,871,85,9.76


In [101]:
volume_threshold = source_summary["applications"].median()
conversion_threshold = source_summary["conversion_pct"].median()

volume_threshold, conversion_threshold

(1048.0, 7.79)

In [102]:
source_summary["quadrant"] = np.select(
    [
        (source_summary["applications"] >= volume_threshold) &
        (source_summary["conversion_pct"] >= conversion_threshold),

        (source_summary["applications"] >= volume_threshold) &
        (source_summary["conversion_pct"] < conversion_threshold),

        (source_summary["applications"] < volume_threshold) &
        (source_summary["conversion_pct"] >= conversion_threshold),

        (source_summary["applications"] < volume_threshold) &
        (source_summary["conversion_pct"] < conversion_threshold)
    ],
    [
        "High volume + high conversion",
        "High volume + low conversion",
        "Low volume + high conversion",
        "Low volume + low conversion"
    ],
    default=""
)

source_summary

,source,applications,hires,conversion_pct,quadrant
0,Agency / RPO,628,40,6.37,Low volume + low conversion
1,Company Website,1225,104,8.49,High volume + high conversion
2,Glassdoor,700,43,6.14,Low volume + low conversion
3,Handshake,387,36,9.30,Low volume + high conversion
4,Indeed,1593,113,7.09,High volume + low conversion
5,LinkedIn,3201,225,7.03,High volume + low conversion
6,Referral,1895,209,11.03,High volume + high conversion
7,University / Campus,871,85,9.76,Low volume + high conversion


### Sourcing Channel Analysis (Scatter Plot)

In general:\
Quadrants based on 1000 applications & 7.8% application-to-hire conversion\
(1,000 applications as the volume threshold, ~7.8% conversion as the conversion threshold)


1. High volume + high conversion - Referral / Company Website
- Referral: 1,895 applications / 11.03% conversion
- Company Website: 1,225 applications / 8.49% conversion
- **Takeaway**: strong combination of reach and hiring efficiency, worth maintaining and potentially investing in
2. High volume + low conversion - LinkedIn / Indeed
- LinkedIn: 3,201 applications / 7.03%
- Indeed: 1,593 applications / 7.09%
- **Takeaway**: strong reach but lower conversion - opportunity to investigate candidate quality, targeting or funnel efficiency
3. Low volume + high conversion - Handshake / University-Campus
- **Takeaway**: relatively efficient channels with limited scale -  worth investigating whether this performance holds across different job levels
4. Low volume + low conversion - Glassdoor / Agency/RPO
- **Takeaway**: weaker on both reach and hiring conversion - lower priority unless there is a strategic reason to maintain these channels


Conclusion:\
It is the trade-off between volume and hiring efficiency\
Referral stands out as the strongest overall channel, while LinkedIn represents the biggest opportunity for investigation because it generates the most applications but converts at a relatively lower rate

In [103]:
import plotly.express as px

fig = px.scatter(
    source_summary,
    x="applications",
    y="conversion_pct",
    text="source",
    size="hires",
    hover_data=["applications", "hires", "conversion_pct", "quadrant"],
    title="Sourcing Channels: Volume vs Hiring Conversion",
    labels={
        "applications": "Application Volume",
        "conversion_pct": "Application-to-Hire Conversion (%)",
        "hires": "Hires"
    }
)

fig.add_vline(
    x=volume_threshold,
    line_dash="dash"
)

fig.add_hline(
    y=conversion_threshold,
    line_dash="dash"
)

fig.update_traces(
    textposition="top center"
)

fig.show()

## Priority III - Recruiter

### 💾 解決方案：從 Google Drive 自動讀取資料 (免重複上傳)

建議將您的 `recruiting_funnel_data.csv` 上傳到您的 Google 雲端硬碟 (例如放在 `My Drive` 底下)。
接著，執行下方的程式碼來掛載雲端硬碟並讀取檔案。第一次執行時，Colab 會要求您授權存取雲端硬碟。

In [105]:
recruiter_summary = (
    df.groupby("recruiter")
      .agg(
          applications=("candidate_id", "count"),
          hires=("reached_hired", "sum")
      )
      .reset_index()
)

recruiter_summary["conversion_pct"] = (
    recruiter_summary["hires"]
    / recruiter_summary["applications"]
    * 100
).round(2)

recruiter_summary.sort_values(
    "conversion_pct",
    ascending=False
)

,recruiter,applications,hires,conversion_pct
3,Jordan Walsh,1314,120,9.13
7,Taylor Kim,1291,113,8.75
6,Sam Rivera,1365,114,8.35
5,Riley Johnson,1309,109,8.33
2,Drew Martinez,1374,112,8.15
0,Alex Patel,1287,100,7.77
1,Casey Chen,1261,97,7.69
4,Morgan Lee,1299,90,6.93


###Finding I:

recruiter conversion varies but the data hasn't established whether this reflects recruiter performance or differences in hiring mix

In [ ]:
recruiter_job_summary = (
    df.groupby(["recruiter", "job_level"])
      .agg(
          applications=("candidate_id", "count"),
          hires=("reached_hired", "sum")
      )
      .reset_index()
)

recruiter_job_summary["conversion_pct"] = (
    recruiter_job_summary["hires"]
    / recruiter_job_summary["applications"]
    * 100
).round(2)

recruiter_job_summary

In [ ]:
recruiter_job_summary.sort_values(
    ["job_level", "conversion_pct"],
    ascending=[True, False]
)

In [ ]:
recruiter_job_pivot = recruiter_job_summary.pivot(
    index="recruiter",
    columns="job_level",
    values="conversion_pct"
).round(2)

recruiter_job_pivot

### Finding II:

Recruiter performance varies considerably by job level, suggesting that overall recruiter conversion rates alone may not provide a complete picture of performance.

Some recruiters perform strongly for specific job levels but not others. This indicates that differences in recruiter conversion may reflect role mix, candidate pools, hiring complexity, or recruiter–role fit, rather than individual recruiter performance alone.

Implication: Recruiter performance should be evaluated within comparable job levels before using conversion rate as a performance indicator.

In [107]:
recruiter_job_summary = (
    df.groupby(["recruiter", "job_level"])
      .agg(
          applications=("candidate_id", "count"),
          hires=("reached_hired", "sum")
      )
      .reset_index()
)

recruiter_job_summary["conversion_pct"] = (
    recruiter_job_summary["hires"]
    / recruiter_job_summary["applications"]
    * 100
).round(2)

recruiter_job_volume = recruiter_job_summary.pivot(
    index="recruiter",
    columns="job_level",
    values="applications"
)

recruiter_job_volume

job_level,IC1,IC2,IC3,IC4,IC5,IC6,M1,M2,M3,M4
recruiter,,,,,,,,,,
Alex Patel,82,178,183,197,123,74,74,181,159,36
Casey Chen,88,160,193,177,122,56,90,159,161,55
Drew Martinez,86,194,186,206,134,68,90,182,179,49
Jordan Walsh,90,158,198,176,108,97,83,181,171,52
Morgan Lee,65,177,197,184,133,68,76,173,171,55
Riley Johnson,82,158,219,171,107,79,82,165,195,51
Sam Rivera,79,169,200,188,117,83,101,202,182,44
Taylor Kim,108,168,188,197,101,77,86,167,146,53


In [108]:
(recruiter_job_volume >= 50).sum().sum()

np.int64(77)

In [109]:
(recruiter_job_volume < 50).sum().sum()

np.int64(3)

In [110]:
recruiter_job_reliable = recruiter_job_summary.copy()

recruiter_job_reliable["conversion_reliable"] = np.where(
    recruiter_job_reliable["applications"] >= 50,
    recruiter_job_reliable["conversion_pct"],
    np.nan
)

recruiter_job_reliable

,recruiter,job_level,applications,hires,conversion_pct,conversion_reliable
0,Alex Patel,IC1,82,9,10.98,10.98
1,Alex Patel,IC2,178,11,6.18,6.18
2,Alex Patel,IC3,183,26,14.21,14.21
3,Alex Patel,IC4,197,9,4.57,4.57
4,Alex Patel,IC5,123,9,7.32,7.32
...,...,...,...,...,...,...
75,Taylor Kim,IC6,77,7,9.09,9.09
76,Taylor Kim,M1,86,7,8.14,8.14
77,Taylor Kim,M2,167,19,11.38,11.38
78,Taylor Kim,M3,146,13,8.90,8.90


In [111]:
np.nan

nan

### Recruiter × Job-level (Heatmap)



In [112]:
heatmap_data = recruiter_job_reliable.pivot(
    index="recruiter",
    columns="job_level",
    values="conversion_reliable"
)

fig = px.imshow(
    heatmap_data,
    text_auto=".1f",
    aspect="auto",
    labels={
        "x": "Job Level",
        "y": "Recruiter",
        "color": "Conversion (%)"
    },
    title="Recruiter × Job Level: Hiring Conversion (≥50 Applications)"
)

fig.update_traces(
    hovertemplate=
    "Recruiter: %{y}<br>" +
    "Job Level: %{x}<br>" +
    "Conversion: %{z:.2f}%<extra></extra>"
)

fig.update_layout(
    width=1100,
    height=600
)

fig.show()